# 第1章：数字图像的获取和表示

## 编程实践：Gamma 校正

---

## 一、什么是数字图像？

### 1.1 图像的本质

数字图像是通过**相机拍摄三维物理世界**记录的**平面投影**。它本质上是一个**二维矩阵**，矩阵中的每个元素称为**像素（Pixel）**，每个像素存储一个数值，表示该位置的颜色或亮度。

```
一幅图像 = 一个 M×N 的像素矩阵

  ┌────────────────────────────┐
  │ 像素(0,0) │ 像素(1,0) │ ... │
  │ 像素(0,1) │ 像素(1,1) │ ... │
  │    ...    │    ...    │ ... │
  └────────────────────────────┘
       M 行 × N 列
```

### 1.2 像素值的含义

- **灰度图像**：每个像素用一个数值表示亮度，范围 0~255（8位）
  - 0 表示黑色（最暗），255 表示白色（最亮）
- **彩色图像**：每个像素用三个数值表示颜色，通常是 R、G、B 三个通道
  - OpenCV 使用 **BGR** 顺序（蓝、绿、红）
  - 每个通道范围也是 0~255

### 1.3 图像的获取流程（ISP 管线）

数码相机的成像管线包含以下环节：

```
光线 → 镜头 → 光圈 → 快门 → 感光器(CCD/CMOS) → 模数转换(ADC) → ISP处理 → 数字图像
```

| 环节 | 作用 |
|------|------|
| 镜头 | 聚焦光线，形成光学图像 |
| 光圈 | 控制进光量 |
| 快门 | 控制曝光时间 |
| 感光器 | 将光信号转为电信号 |
| ADC | 模拟信号→数字信号 |
| ISP | 去马赛克、降噪、色彩校正、Gamma校正等 |

### 1.4 像素矩阵的特点

- **离散性**：像素值是离散的（整数），不是连续的
- **有限性**：矩阵大小有限（如 1920×1080）
- **量化误差**：连续的物理世界被量化为 256 个等级
- **空间相关性**：相邻像素通常具有相似的值


## 二、什么是 Gamma 校正？

### 2.1 定义

Gamma 校正是一种**非线性**的图像调整技术，用于改变图像的亮度和对比度。它基于**幂函数**对像素值进行变换。

### 2.2 为什么需要 Gamma 校正？

1. **人眼视觉特性**：人眼对光线强度的感知是**非线性**的（对数关系），而显示设备的光强输出是**线性**的。Gamma 校正可以使图像显示更符合人眼感知。
2. **显示器校准**：补偿不同显示器的 Gamma 响应曲线。
3. **图像增强**：提亮暗部细节或压缩过亮部分。

### 2.3 数学原理

```
公式：output = input^(1/gamma)

其中：
  input  — 归一化后的像素值 (0~1)
  gamma  — 控制曲线形状的参数
  output — 校正后的像素值
```

### 2.4 Gamma 值的影响

| Gamma 值 | 效果 | 说明 |
|-----------|------|------|
| gamma > 1 | 图像变亮 | 暗部被提亮 |
| gamma < 1 | 图像变暗 | 亮部被压缩 |
| gamma = 1 | 无变化 | 恒等变换 |
| gamma = 2.2 | 典型值 | CRT 显示器 gamma |

### 2.5 变换曲线示意

```
输出 1.0 ┤                    ╱ gamma=0.5(变暗)
         │                 ╱
         │              ╱
    1.0 ┼─────────────╱───────── gamma=1.0(不变)
         │         ╱
         │      ╱
    0.0 ┼─────╱  gamma=2.2(变亮)
         └────────────────── 输入
         0.0              1.0
```

### 2.6 应用场景
- 显示器色彩校准
- 暗部图像增强（夜景、室内照片）
- 数字摄影后期处理
- 医学图像处理


## 三、实现要求

> **要求**：使用 Python 调用 OpenCV 的图像读取/保存函数，对彩色图像进行 Gamma 校正。除 OpenCV 的读写函数外，其余代码全部手写，不能调用其他函数库。

### 约束条件
- ✅ 可以使用 `cv2.imread()` 读取图像
- ✅ 可以使用 `cv2.imwrite()` 保存图像
- ❌ 不能使用 `cv2.LUT()` 等 OpenCV 图像处理函数
- ❌ 不能使用 `numpy.power()` 等向量化运算
- ❌ 不能使用 PIL、skimage 等其他库
- ✅ 可以使用 Python 基本语法（循环、条件、数学运算）

### 核心思路
```
读取图像 → 逐像素遍历 → 归一化 → Gamma公式 → 反归一化 → 保存图像
```


In [ ]:
# 导入 OpenCV 库（仅用于图像读写）
import cv2

print(f"OpenCV 版本: {cv2.__version__}")

In [ ]:
# 生成测试图像 (运行时自动生成)
import numpy as np

print("正在生成测试图像...")
h, w = 400, 600
test_img = np.zeros((h, w, 3), dtype=np.uint8)
for y in range(h):
    for x in range(w):
        test_img[y, x, 0] = int(180 * x / w)
        test_img[y, x, 1] = int(200 * y / h)
        test_img[y, x, 2] = int(120 + 80 * (x + y) / (w + h))
cv2.rectangle(test_img, (50, 50), (150, 150), (0, 0, 255), -1)
cv2.rectangle(test_img, (200, 50), (300, 150), (0, 255, 0), -1)
cv2.rectangle(test_img, (350, 50), (450, 150), (255, 0, 0), -1)
cv2.circle(test_img, (150, 250), 60, (255, 0, 255), -1)
cv2.circle(test_img, (350, 280), 80, (128, 255, 128), -1)
cv2.putText(test_img, "Color Image", (180, 380), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
cv2.imwrite("color_image.jpg", test_img)
print("测试图像已生成: color_image.jpg")

In [ ]:
def gamma_correction_manual(image, gamma):
    """
    手写实现 Gamma 校正函数
    
    参数:
        image: 输入图像 (OpenCV读取的BGR格式图像, uint8类型)
        gamma: Gamma值 (float)
               - gamma > 1: 提亮暗部
               - gamma < 1: 压暗暗部
               - gamma = 1:  无变化
    
    返回:
        校正后的图像 (uint8类型)
    """
    
    # ========== 步骤1: 获取图像信息 ==========
    # image.shape 返回 (height, width, channels)
    # 注意: OpenCV 中彩色图像是 BGR 顺序, 不是 RGB
    height, width, channels = image.shape
    
    # ========== 步骤2: 创建结果图像 ==========
    # 先复制原图, 转为 float32 以支持小数运算
    result = image.copy().astype('float32')
    
    # ========== 步骤3: 计算 gamma 倒数 ==========
    # 公式: output = input^(1/gamma)
    # gamma>1 时, 1/gamma<1, 暗值变大(提亮)
    # gamma<1 时, 1/gamma>1, 暗值变小(压暗)
    inv_gamma = 1.0 / gamma
    
    # ========== 步骤4: 逐像素进行 Gamma 变换 ==========
    
    for y in range(height):          # 遍历每一行
        for x in range(width):      # 遍历每一列
            for c in range(channels):  # 遍历 B/G/R 三个通道
                
                # Step 4.1: 获取当前像素值 (范围 0~255)
                pixel_value = result[y, x, c]
                
                # Step 4.2: 归一化到 [0, 1] 区间
                # 将 [0, 255] 映射到 [0.0, 1.0]
                normalized = pixel_value / 255.0
                
                # Step 4.3: 应用 Gamma 公式
                # output = input^(1/gamma)
                gamma_corrected = normalized ** inv_gamma
                
                # Step 4.4: 反归一化回 [0, 255]
                # 将 [0.0, 1.0] 映射回 [0, 255]
                result[y, x, c] = gamma_corrected * 255.0
    
    # ========== 步骤5: 转换回 uint8 ==========
    # OpenCV 要求图像为 uint8 类型
    result = result.astype('uint8')
    
    return result

In [ ]:
# ==================== 配置参数 ====================

# 输入图像路径
input_image_path = "color_image.jpg"

# 输出图像路径
output_image_path = "gamma_corrected.jpg"

# Gamma 值 (可调整观察效果)
gamma_value = 2.2  # 提亮暗部

# ==================== 读取图像 ====================
print("正在读取图像...")

# cv2.imread 参数说明:
# 参数1: 图像文件路径
# 参数2: 读取模式
#   cv2.IMREAD_COLOR (1):     彩色图像 (BGR格式)
#   cv2.IMREAD_GRAYSCALE (0): 灰度图像
#   cv2.IMREAD_UNCHANGED (-1): 含alpha通道
img = cv2.imread(input_image_path, cv2.IMREAD_COLOR)

# 检查图像是否成功读取
if img is None:
    print(f"错误: 无法读取图像 '{input_image_path}'")
else:
    # 打印图像基本信息
    h, w, ch = img.shape
    print(f"图像读取成功!")
    print(f"  - 尺寸: {w} x {h}")
    print(f"  - 通道数: {ch} (BGR格式)")
    print(f"  - 数据类型: {img.dtype}")
    print(f"  - 像素值范围: [{img.min()}, {img.max()}]")

In [ ]:
# ==================== 执行 Gamma 校正 ====================
if img is not None:
    print(f"正在进行 Gamma 校正 (gamma={gamma_value})...")
    print("逐像素处理中, 对于大图像可能需要一些时间...\n")
    
    # 调用手写的 gamma 校正函数
    corrected_img = gamma_correction_manual(img, gamma_value)
    
    # 打印校正后图像的信息
    print(f"校正完成!")
    print(f"  - 校正后像素值范围: [{corrected_img.min()}, {corrected_img.max()}]")
    print(f"  - 校正后平均值: {corrected_img.mean():.2f}")

In [ ]:
# ==================== 保存结果图像 ====================
if img is not None:
    print(f"正在保存结果图像到 '{output_image_path}'...")
    
    # cv2.imwrite 参数说明:
    # 参数1: 保存路径 (扩展名决定格式: .jpg, .png, .bmp等)
    # 参数2: 要保存的图像数组 (必须是 uint8 类型)
    success = cv2.imwrite(output_image_path, corrected_img)
    
    if success:
        print(f"保存成功! 结果已保存到: {output_image_path}")
    else:
        print(f"保存失败! 请检查文件路径和格式。")

In [ ]:
# ==================== 可视化对比 ====================
import matplotlib.pyplot as plt

# 读取原图和校正后图像
original = cv2.imread(input_image_path)
corrected = cv2.imread(output_image_path)

if original is not None and corrected is not None:
    # OpenCV 是 BGR 格式, matplotlib 需要 RGB 格式
    # 所以需要转换通道顺序
    original_rgb = cv2.cvtColor(original, cv2.COLOR_BGR2RGB)
    corrected_rgb = cv2.cvtColor(corrected, cv2.COLOR_BGR2RGB)
    
    # 创建对比图
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    axes[0].imshow(original_rgb)
    axes[0].set_title('Original Image')
    axes[0].axis('off')
    
    axes[1].imshow(corrected_rgb)
    axes[1].set_title(f'Gamma Corrected (gamma={gamma_value})')
    axes[1].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    # 像素值分布对比
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    channels = ['B', 'G', 'R']
    
    for i, (channel_name, channel_idx) in enumerate(zip(channels, range(3))):
        axes[i].hist(original[:,:,channel_idx].ravel(), bins=64, alpha=0.5, 
                    label='Original', color='blue')
        axes[i].hist(corrected[:,:,channel_idx].ravel(), bins=64, alpha=0.5, 
                    label='Corrected', color='red')
        axes[i].set_title(f'{channel_name} Channel Histogram')
        axes[i].legend()
    
    plt.tight_layout()
    plt.show()

In [ ]:
# ==================== 公式验证 ====================
if original is not None and corrected is not None:
    print("=== 公式验证 ===\n")
    
    # 取一个像素进行公式验证
    h, w = 100, 200
    
    print(f"验证位置: 行={h}, 列={w}")
    print(f"{'通道':<8} {'原值':<8} {'归一化':<10} {'公式结果':<10} {'实际值':<8}")
    print("-" * 55)
    
    for c, name in enumerate(['B', 'G', 'R']):
        orig_val = original[h, w, c]
        norm_val = orig_val / 255.0
        formula_val = (norm_val ** (1.0/gamma_value)) * 255.0
        actual_val = corrected[h, w, c]
        
        print(f"{name:<8} {orig_val:<8} {norm_val:<10.4f} {formula_val:<10.1f} {actual_val:<8}")
    
    # 统计信息
    print(f"\n原图 - 平均值: {original.mean():.2f}, 范围: [{original.min()}, {original.max()}]")
    print(f"校正后 - 平均值: {corrected.mean():.2f}, 范围: [{corrected.min()}, {corrected.max()}]")
    
    if corrected.mean() > original.mean():
        print(f"\n结论: gamma={gamma_value} 使图像变亮 ✓")
    else:
        print(f"\n结论: gamma={gamma_value} 使图像变暗 ✓")

## 四、代码总结

### 完整流程
```
读取图像 → 获取维度 → 转float → 逐像素遍历 → 归一化
    → Gamma公式(input^(1/gamma)) → 反归一化 → 转uint8 → 保存
```

### 关键代码解析

| 步骤 | 代码 | 说明 |
|------|------|------|
| 获取维度 | `image.shape` | 返回 (高, 宽, 通道数) |
| 归一化 | `pixel / 255.0` | [0,255] → [0,1] |
| Gamma变换 | `normal ** inv_gamma` | 幂函数运算 |
| 反归一化 | `result * 255.0` | [0,1] → [0,255] |
| 类型转换 | `.astype('uint8')` | 转回图像格式 |

### 注意事项
1. **OpenCV 通道顺序是 BGR，不是 RGB**
2. **像素值范围是 [0, 255]，需要归一化**
3. **逐像素三重循环较慢，但本题要求手写**
4. **JPEG 有损压缩会引入微小误差**
5. **gamma 值建议在 0.1~10 范围内**

### 扩展练习
1. 尝试不同的 gamma 值 (0.5, 1.0, 3.0) 观察效果
2. 如何对灰度图像进行 Gamma 校正？（提示：只有2重循环）
3. 为什么公式用 `input^(1/gamma)` 而不是 `input^gamma`？
